# TP5 — Confidence Intervals (CIs)

Estimated time: **1h30**

🎯 **Learning Objectives**

In this lab, you'll go over hands-on exercises using Python to further consolidate your understanding about confidence interval. By the end of this session, you should be able to:
- Understand what a confidence interval (CI) is and what the confidence *level* means.
- Compute CIs for **means**.
- Compute and interpret the **margin of error (MoE)**.
- Understand how **sample size** and **confidence level** affect CI width.

🍄 **Outline**
* Get familiar with the dataset
* Sample exercises
  * Compute the confidence interval and margin of error
  * Investigate the impact of sample size and confidence level on the width of CI
  * Understand intuition behind CI
  * Bootstrap for confidence interval
* Assignment questions


### 📚 Import Libraries


Install required packages for this lab. `pip` will automatically skip already installed packages

In [ ]:
import importlib.util

# import name: pip installation name
required_pkgs = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "statsmodels": "statsmodels",
    "scipy": "scipy",
}

missing_pkgs = [
    pip_name
    for import_name, pip_name in required_pkgs.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_pkgs:
    print("⚠️ Missing packages:", ", ".join(missing_pkgs))
    packages = " ".join(missing_pkgs)
    %pip install $packages
else:
    print("🎉 All packages already installed.")

Import the required packages


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.datasets import load_diabetes

## 📊 Get Familiar with the Dataset
For this TP, we will make use of the **Diabetes** dataset from the scikit-learn library. 
It contains **442 patients** with 10 baseline variables and a progression variable which measures disease progression one year after baseline.

**It contains the following columns**:
- `age` — Age in years  
- `sex` — Sex indicator (1.0 = male, 2.0 = female)  
- `bmi` — Body Mass Index (kg/m²)  
- `bp` — Average blood pressure (mmHg)  
- `s1 - s6` — Different standardized blood serum measurement 
- `progression` — Diabetes **progression** score (higher = worse) one year later

For the following exercise, we will mainly make use of the **age**, **sex**, **bmi**, **bp**, **progression** columns. 


**Load the dataset**

In [ ]:
from sklearn.datasets import load_diabetes

# Load the diabetes dataset as a pandas DataFrame
# Parameters: as_frame=True: return the data in a pandas DataFrame instead of a NumPy array
#             scaled=False: load the original (unstandardized) values instead of normalized ones
diab = load_diabetes(as_frame=True, scaled=False)
df = diab.frame.copy()

# Rename the column "target" to "progression" to make it more intuitive to understand
df = df.rename(columns={"target": "progression"})

# Display the first 5 rows of the dataset
df.head()

**Display the size of the dataset**

In [ ]:
df.shape

**Display summary statistics for each column of data**

In [ ]:
# .round(2) keeps 2 digits after the decimal point
df.describe().round(2)

Visualize the dataset using histograms, with a focus on **age**, **bmi**, **bp**, **progression**

In [ ]:
# Select the columns we want to plot
cols = ['age', 'bmi', 'bp', 'progression']

# Set up a 1x4 grid of subplots
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Plot a histogram for the interested column of data
for ax, col in zip(axes.flatten(), cols):
    sns.histplot(df[col], bins=10, kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()


# 📝 Example Exercises

## <span style="color:blue">  Example-1: formula-based confidence interval

**Task: draw one sample and compute the sample mean, the estimated standard error and the 95% confidence interval** </span>


Given a sample, we can use the following formula to compute a confidence interval for the population mean, if the population standard deviation is unknown

$$
\Large
\bar{x}\pm t_{n-1,\,1-\alpha/2}\frac{s}{\sqrt n}.
$$

where,
- $\bar{x}$ is the estimate, e.g. sample mean
- $s$ is the standard deviation of the sample
- $n$ is the sample size
- $\frac{s}{\sqrt n}$ is the estimated standard error
- $t_{n-1,\,1-\alpha/2}$ is the critical value that depends on degrees of freedom $n-1$ and the confidence level ($\alpha=0.05$ for confidence level 95%)

<span style="color:blue"> **Draw one random sample and compute the sample mean** </span>

In [ ]:
# Set a random seed for reproducibility
rng = np.random.default_rng(seed=2026)
sample_size = 100

# Take a random sample of 100 observations without replacement from the dataset.
bmi = df['bmi']
one_bmi_sample = bmi.sample(n=sample_size,      # sample size
                            replace=False,      # sampling without replacement 
                            random_state=rng    # random seed for reproducibility
                                     )

# Compute the sample mean
bmi_sample_mean = one_bmi_sample.mean()
print(f"Sample mean of the random BMI sample is: {bmi_sample_mean:,.2f}")

In the above code, we take a random sample of 100 BMI observations and compute the sample mean.

<span style="color:blue"> **Compute the estimated standard error (SE)** </span>

The **estimated standard error** can be computed based on the sample as below, where `s` is the sample standard deviation, `n` is the sample size

$$
\Large
\widehat{SE} = \frac{s}{\sqrt n}.
$$



In [ ]:
# Compute the standard deviation (SD) of the sample. ddof=1 applies the
# usual n-1 denominator for the sample SD.
bmi_sample_std = one_bmi_sample.std(ddof=1)

# Estimated standard error of the sample mean.
bmi_sample_se = bmi_sample_std / np.sqrt(len(one_bmi_sample))

print(f"The estimated standard error is: {bmi_sample_se:,.2f}")

<span style="color:blue"> **Compute the margin of error and the 95% confidence interval of the sample mean** </span>

The **margin of error (MoE)** can be computed as

$$
\Large
MoE = t_{n-1,\,1-\alpha/2} * \widehat{SE} 
$$

And then the **confidence interval (CI)** can be represented as: 

$$
\Large
CI = \bar{x} \pm {MoE}
$$

In [ ]:
# Compute the two-sided t critical value for the confidence level.
# A two-sided interval leaves alpha/2 probability in each tail. 
# For a one-sample t interval, the number of degrees of freedom is n - 1.
confidence_level = 0.95
alpha = 1 - confidence_level
critical = stats.t.ppf(1 - alpha/2, df= len(one_bmi_sample) - 1)

# Compute the margin of error = critical value × SE, and the
# two bounds for the confidence interval.
margin_of_error = critical * bmi_sample_se
lower_bound = bmi_sample_mean - margin_of_error
upper_bound = bmi_sample_mean + margin_of_error

print(f"Sample size n = {len(one_bmi_sample)}")
print(f"Sample mean (BMI) = {bmi_sample_mean:.2f}")
print(f"Estimated standard error = {bmi_sample_se:.2f}")
print(f"The 95% confidence interval for the average bmi is: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Margin of error (95% CI) = {margin_of_error:.2f} BMI units")

The confidence interval computed based on the sample suggests that the **population mean BMI** is estimated to lie between 25.43 and 27.16, with 95% confidence.

The **margin of error (0.86)** tells us how far the sample mean might be from the population mean due to sampling variation.  In other words, our sample mean is an estimate that could be off by about ±0.86 BMI units.

A two-sided 95% confidence interval leaves 5% outside the interval: 2.5% in each tail. `stats.t.ppf(0.975, df=99)` finds the 97.5th percentile of the t-distribution with \(n-1=99\) degrees of freedom. This gives the positive critical value $t^*$ needed to capture the central 95% of the distribution.


<span style="color:blue"> **Build a dataframe to display the different values** </span>

In [ ]:
result = {
        "n": len(one_bmi_sample),
        "sample mean": bmi_sample_mean,
        "standard deviation": bmi_sample_std,
        "standard error": bmi_sample_se,
        "confidence level": confidence_level,
        "critical value": critical,
        "MoE": margin_of_error,
        "lower bound": lower_bound,
        "upper bound": upper_bound,
    }

result = pd.DataFrame([result]).round({
        "sample mean": 2, "standard deviation": 2, "standard error": 2, "critical value": 2,
        "MoE": 2, "lower bound": 2, "upper bound": 2,
    })

result

## <span style="color:blue"> Example-2: Effect of Confidence Level on CI Width </span>

**Tasks**: The goal is to compute the 90%, 95%, 99% CIs for the mean BMI based on a sample, and to see how does confidence level affect the width of the confidence interval.

<span style="color:blue"> **Compute the 90%, 95%, 99% CIs for the mean BMI and diplay the results in a well-formatted Pandas dataframe** </span>

In [ ]:
# Calculating the mean and the estimated standard error
bmi_sample_mean = one_bmi_sample.mean()
bmi_sample_std = one_bmi_sample.std(ddof=1)
bmi_sample_size = len(one_bmi_sample)
bmi_sample_se = bmi_sample_std / np.sqrt(len(one_bmi_sample))

confidence_levels = [0.90, 0.95, 0.99]
bmi_confidence_intervals = []

# Calculating confidence intervals for 90%, 95%, and 99%
for confidence_level in confidence_levels:
    # determine the critical value for each confidence level
    alpha = 1 - confidence_level
    critical = stats.t.ppf(1 - alpha / 2, df= bmi_sample_size - 1)
    
    # Compute the margin of error = critical value × SE, and the two  
    # bounds for the confidence interval. Since it is based on the 
    # sample sample, the estimated standard error is the same.
    margin_of_error = critical * bmi_sample_se
    lower_bound = bmi_sample_mean - margin_of_error
    upper_bound = bmi_sample_mean + margin_of_error
    bmi_confidence_intervals.append((confidence_level, lower_bound, upper_bound, margin_of_error))

# Convert to DataFrame for better visualization. round(2): keep 2 digits after the decimal points
bmi_results = pd.DataFrame(bmi_confidence_intervals, columns=["Confidence Level", "Lower bound", "Upper bound", "Margin of Error"]).round(2)
bmi_results

<span style="color:blue"> **Visualize the 90%, 95%, 99% CIs for the mean BMI. A verticle line which represents the sample mean BMI is also incldued.** </span>

In [ ]:
# Plotting the confidence intervals with BMI on the x-axis and Confidence Level on the y-axis
plt.figure(figsize=(9, 5))

plt.errorbar(
    # Creates a list with the same BMI mean value repeated for each confidence level 
    x=[bmi_sample_mean] * len(bmi_confidence_intervals), y=['90%', '95%', '99%'],
    # The size of the error bar is the margin of error 
    xerr=[moe for cl, lower_bound, upper_bound, moe in bmi_confidence_intervals], 
    # Display the mean as a point (fmt=o), display the markers and error bars in blue (color='b'),  
    # specify the width of the "caps" at the end of the error bars (capsize=10)
    fmt='o', color='b', capsize=10
)

# Adding a vertical red dashed-line for the sample mean
plt.axvline(x=bmi_sample_mean, color='r', linestyle='--', label=f'Mean BMI: {bmi_sample_mean:.2f}')
plt.legend(loc='upper right', bbox_to_anchor=(0.90, 0.90))

# Adding labels and title
plt.title('Confidence Intervals for BMI at 90%, 95%, 99% Confidence Level')
plt.xlabel('BMI')
plt.ylabel('Confidence Level')

plt.show()

❓<span style="color:purple;">**Quick test**</span>:  according to the above plot, **does the confidence interval become wider or narrower as we increase the confidence level (from 90% → 95% → 99%)?**

<details>
<summary>Click to reveal the answer</summary>

**Answer:**  
<p style="color:blue;">
The confidence interval becomes <b>wider</b> as the confidence level increases.  
</p>

**Explanation:**  
<p style="color:blue;">
A higher confidence level (e.g., 99%) means we want to be more certain that the true population mean lies inside the interval. To achieve this greater certainty, we must allow for a larger "margin of error."  As a result, the interval expands (widens).  
</p>
<p style="color:blue;">
Conversely, a lower confidence level (e.g., 90%) gives a narrower interval but with less certainty.  
</p>

<p style="color:green;">
(<b>for advanced students only</b>) Mathematically, a two-sided confidence interval is calculated as \(\bar{x} \pm t^* \times SE\), so its total width is \(2 \times t^* \times SE\). For the same sample, the sample mean and SE remain unchanged. However, the critical value \(t^*\) increases as the confidence level increases. For example, for a sample size n=100 (which gives 99 degrees of freedom in critical value calculation), the critical values are approximately 1.66, 1.98, and 2.63 for the 90%, 95%, and 99% intervals, respectively. Therefore, the margin of error—and consequently the interval width—increases.
</p>

</details>


## <span style="color:blue"> Example-3: Effect of Sample Size on CI Width </span>

According to the formula,  the sample size $n$ also has a direct impact on the confidence interval. For a fixed confidence level, the width of the CI is approximately proportional to $\frac{1}{\sqrt{n}}$. Consequently, **to halve the width of the CI, the sample size needs to be increased by a factor of 4**.

$$
\Large
\bar{x}\pm t_{n-1,\,1-\alpha/2}\frac{s}{\sqrt n}.
$$

<span style="color:blue"> **Below, we compute the CI for 3 different sample sizes: 25, 50, 100 for 95% confidence level**

In [ ]:
# Define 3 different sample sizes and 
# the confidence level
sample_sizes = [25, 50, 100]
confidence_level = 0.95

# Set a random seed for reproducibility
rng = np.random.default_rng(seed=2026)

# The dictionary will hold the mean, the 95% CI and 
# the MoE for each sample size.
ci_for_varying_sizes = {}

# Treat all the bmi values as our population
initial_sample = df["bmi"]

# Compute the 95% CI for each sample size
for sample_size in sample_sizes:
    # Take a random sample of different size without replacement from the dataset.
    one_random_sample = initial_sample.sample(n=sample_size,   # sample size
                                              replace=False,       # sampling without replacement 
                                              random_state=rng    # random seed for reproducibility
                                           )    
    
    # Calculating the mean and the estimated standard error
    bmi_sample_mean = one_random_sample.mean()
    bmi_sample_std = one_random_sample.std(ddof=1)
    bmi_sample_se = bmi_sample_std / np.sqrt(sample_size)
    
    # determine the critical value 
    alpha = 1 - confidence_level
    critical = stats.t.ppf(1 - alpha / 2, df= sample_size - 1)
    
    # Compute the margin of error = critical value × SE, and the
    # two bounds for the confidence interval, as well as the 
    # confidence interval width for comparison.
    margin_of_error = critical * bmi_sample_se
    lower_bound = bmi_sample_mean - margin_of_error
    upper_bound = bmi_sample_mean + margin_of_error
    ci_for_varying_sizes[sample_size] = (bmi_sample_mean, lower_bound, upper_bound, margin_of_error)

# Convert to DataFrame for better visualization. round(2): keep 2 digits after the decimal points
# orient="index" uses the dictionary keys—the sample sizes—as the row index
# rename_axis() labels the row index as "Sample Size".
bmi_results = pd.DataFrame.from_dict(
    ci_for_varying_sizes,
    orient="index",
    columns=["Mean", "Lower", "Upper", "MoE"],
).rename_axis("Sample Size").round(2)


bmi_results

❓<span style="color:purple;">**Quick test**</span>: According to the above table, when the sample size increases from **25** to **100**, what happens to the width of the confidence interval?

**A.** The sample size increases by a factor of 4, and the interval width also increases by a factor of 4.

**B.** The sample size increases by a factor of 4, and the interval width decreases to approximately one-half of its original width.

**C.** The sample size increases by a factor of 4, but the interval width remains approximately the same.

<details>
<summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;">
<strong>B. The interval width decreases to approximately one-half of its original width.</strong>
</p>

**Explanation**:

<p style="color:blue;">
The width of the confidence interval is twice the margin of error. As we can see, when the sample size increases from 25 to 100, the MoE decreases from 1.63 to 0.80 - approximately a factor of 2. Thus, when the sample size increases by a factor of 4, the interval becomes approximately half as wide.
</p>

<p style="color:blue;">
This is consistent with the general relationship
\[
\text{CI width}\propto\frac{1}{\sqrt{n}}.
\]
Because \(\sqrt{4}=2\), multiplying the sample size by 4 reduces the interval width by approximately a factor of 2.
</p>

</details>

❓<span style="color:purple;">**Optional challenge**: </span> Based on the results in the above table, which interpretation is **WRONG**?

**A.** The sample means may differ across samples because of sampling variation.

**B.** Increasing the sample size generally reduces the margin of error and produces a narrower confidence interval.

**C.** Increasing the sample size from 25 to 100 must reduce the width of the confidence interval to exactly one-half of its original value.


<details>
<summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;">
<b>C. Increasing the sample size from 25 to 100 must reduce the width of the confidence interval to exactly one-half of its original value.</b>
</p>

**Explanation**:

<p style="color:blue;">
The width of the confidence interval is twice the margin of error, which is calculated as $\text{MoE} = t_{n-1, 1-\alpha/2} \frac{s}{\sqrt{n}}$ and is approximately proportional to \( \tfrac{1}{\sqrt{n}} \). Therefore, increasing the sample size from 25 to 100 (a factor of 4) should reduce the margin of error by approximately a factor of \( \sqrt{4}=2 \).
</p>

<p style="color:blue;">
However, the reduction may not be exact because the sample standard deviation $s$ and the critical value $t$ can also vary. 
</p>

</details>

<span style="color:blue"> **Visualize the 95% CIs for the mean BMI for each sample size.** </span>

In [ ]:
# Print the results
for n, (mean, lb, ub, moe, *_) in ci_for_varying_sizes.items():
    print(f"n={n}: mean={mean:.2f}, 95% CI = [{lb:.2f}, {ub:.2f}], MoE={moe:.2f}")

# Visualize the CIs for each sample size
fig, ax = plt.subplots(figsize=(9, 5))

for i, (n, (mean, lb, ub, moe, *_)) in enumerate(ci_for_varying_sizes.items()):
    # xerr=moe: set the horizontal error bar length to margin of error (MoE)
    # fmt='o': marker style - 'o' means plot a circular marker for the mean
    # label: text for the legend
    ax.errorbar(x=mean, y=i, xerr=moe, fmt='o', capsize=8, label=f"n={n}, mean={mean:.2f}, MoE={moe:.2f}")

ax.set_yticks(range(len(ci_for_varying_sizes)))
ax.set_yticklabels([f"n={n}" for n in ci_for_varying_sizes.keys()])
ax.set_xlabel("BMI")
ax.set_title("95% CI for Mean BMI for Different Sample Sizes")
ax.legend()
plt.tight_layout()
plt.show()

>As shown on the above plot, as the **sample size increases**, the confidence interval generally becomes **narrower**. When the sample size increases fourfold from **n=25** to **n=100**, the margin of error decreases from **1.63** to **0.80**, which is approximately one-half of its original value.

❓<span style="color:purple;">**Quick test**</span>:
If we want to reduce the width of the confidence interval to approximately **one third of its original width**, how should the sample size change?

<details> <summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;"> The sample size needs to be increased by approximately a factor of <b>9</b>. </p>

**Explanation**:

<p style="color:blue;"> 
    The margin of error is approximately proportional to \( \tfrac{1}{\sqrt{n}} \). Hence, to reduce the width of confidence interval by about a factor of 3, the sample size \(n\) needs to be \(3^2 = 9\) times larger. 
</p> 
<p style="color:blue;"> 
    Hence, the new sample size would need to be approximately \( 9 * n \)
</p> 

</details>

## Example-4: Understanding Intuition of Confidence Intervals through Simulation

### <span style="color:blue;">1) Intuition and Idea </span>

<div style="margin-left:20px">
    
A **confidence interval (CI)** gives us a range of plausible values for a population parameter (such as the mean).

- The **true population mean** is fixed (but usually unknown).

- When we take a sample, we can compute a CI from the sample, for a given confidence level (e.g. 90%, 95%, 99%).

- **If we were to repeat the sampling process many times, the CI would change** from one sample to another (because the sample mean and variability change).

But in the long run, 95% of the obtained CIs will contain the true mean. This is what it means to say we are “95% confident.”

</div>

### <span style="color:blue;"> 2) To give you a more direct idea, we simulate the repeated sampling in Python as follows: </span>

<div style="font-size: 18px; margin-left:15px;">

1. Treat the dataset as the “population”, so the mean of the dataset will be viewed as the population mean
2. Repeatedly take random samples of size 𝑛. 
3. For each sample, compute:
   - sample mean,
   - sample standard deviation,
   - 95% CI for the mean.
   - **Check whether the CI contains the true population mean.**
4. Repeat the random sampling process for many times, and count how often the true mean is covered
5. Visualize the CIs across samples.

</div>

### <span style="color:blue;"> 3) The following code implements the above steps, run it: </span>

In [ ]:
# Simulation settings
sample_size = 40             # sample size per random sampling
number_of_samples = 1000     # number of repeated sampling
confidence_level = 0.95

# Treat all BMI values in the dataset as our population
population = df['bmi']
true_mean = population.mean()

# Set a random seed for reproducibility
rng = np.random.default_rng(seed=2026)

# The list will hold the mean, the 95% CI and the MoE 
# for each sample.
bmi_confidence_intervals = []

#initial_sample = one_bmi_sample
initial_sample = df["bmi"]

# Compute the 95% CI for each sample size
for _ in range(number_of_samples):
    # Take a random sample without replacement from the dataset.
    one_random_sample = initial_sample.sample(n=sample_size,      # sample size
                                              replace=False,       # sampling without replacement 
                                              random_state=rng    # random seed for reproducibility
                                              )    
    
    # Calculating the mean and the estimated standard error
    bmi_sample_mean = one_random_sample.mean()
    bmi_sample_std = one_random_sample.std(ddof=1)
    bmi_sample_se = bmi_sample_std / np.sqrt(len(one_random_sample))
    
    # determine the critical value for each confidence level
    alpha = 1 - confidence_level
    critical = stats.t.ppf(1 - alpha / 2, df= sample_size - 1)
    
    # Compute the margin of error = critical value × SE, and the
    # two bounds for the confidence interval.
    margin_of_error = critical * bmi_sample_se
    lower_bound = bmi_sample_mean - margin_of_error
    upper_bound = bmi_sample_mean + margin_of_error
    # Save the mean, lower bound, upper bound, as well as a boolean result 
    # which indicates whether the confidence interval covers the true mean
    bmi_confidence_intervals.append((bmi_sample_mean, lower_bound, upper_bound, 
                                    lower_bound <= true_mean <= upper_bound))

# Transform the results into DataFrame to ease following processing
df_results = pd.DataFrame(bmi_confidence_intervals, columns=["Mean", "Lower", "Upper", "Covers"])
df_results.head().round(2)

The above table displays the the results computed for the first 5 samples out of 1000 random samples. Each row represents a different random sample:

- **Mean** is the sample mean.
  
- **Lower** and **Upper** are the lower and upper bounds of the confidence interval.
- **Covers** indicates whether the confidence interval contains the true population mean:
  - `True`: the interval contains the population mean.
  - `False`: the interval does not contain the population mean.

In this table, all five confidence intervals cover the true population mean. This does not mean that every confidence interval will do so. With a 95% confidence procedure, approximately 95% of intervals constructed from many repeated samples are expected to contain the true population mean.

<span style="color:blue;"> **Compute the percentage of samples that cover the true mean** </span>

In [ ]:
# Compute how many samples captures the true mean
num_covers = df_results["Covers"].sum()   # how many CIs contain the true mean
total      = len(df_results)              # total number of CIs
coverage   = num_covers / total
print(f"Out of {number_of_samples} samples, {coverage*100:.1f}% of CIs contained the true mean.")

❓<span style="color:purple;">**Quick test**</span>: In the simulation, 95.3% of the 1,000 confidence intervals contained the true population mean. Is this result consistent with a 95% confidence level?

**A.** No. Exactly 95.0% of the intervals must contain the population mean.

**B.** Yes. The observed coverage is close to 95%, and small differences are expected because of random sampling variation.

**C.** No. A 95% confidence level means that every confidence interval must contain the population mean.

<details>
<summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;">
<strong>B. Yes. The result is consistent with a 95% confidence level.</strong>
</p>

**Explanation**:

<p style="color:blue;">
A 95% confidence procedure is expected to produce intervals that contain the true population mean approximately 95% of the time over many repeated samples. It does not require the observed percentage to be exactly 95% in every simulation.
</p>

<p style="color:blue;">
Here, 95.3% of the 1,000 confidence intervals contained the true mean. This is very close to 95%, and the small difference is due to random sampling variation.
</p>

</details>

<span style="color:blue;"> **Plot the confidence intervals for a subset of the 1000 samples. Use different color to indicate whether it covers the true mean**

In [ ]:
# Plot a subset of the intervals
fig, ax = plt.subplots(figsize=(6, 8))

# show first 40 confidence intervals: if the CI covers the true mean, then it will be green.
# Else, we highlight it as red.
subset = df_results.iloc[:40]  
for i, row in subset.iterrows():
    color = "green" if row["Covers"] else "red"
    ax.errorbar(x=row["Mean"], y=i, xerr=[[row["Mean"]-row["Lower"]],[row["Upper"]-row["Mean"]]],
                fmt='o', color=color, capsize=4)

# Plot the true population mean (blue vertical line)
ax.axvline(x=true_mean, color='blue', linestyle='--', label=f"Population mean = {true_mean:.2f}")

ax.set_ylabel("Sample number")
ax.set_xlabel("BMI")
ax.set_title("95% Confidence Intervals across repeated samples")
ax.legend()
plt.tight_layout()
plt.show()

In the above output, the **blue verticle line** corresponds to the true **population mean** (because here we treat the whole dataset as our population). 

Each time we take a random sample from the population, compute the sample mean as well as its confidence interval. **If the confidence interval covers the population mean, then it is plotted in green. Otherwise, it is plotted in red.** 

We repeat the random sampling for 1000 times and display the result for the first 40 samples. 

### <span style="color:blue;"> 4) What observations can we make: </span> 

- The **population mean** (blue dashed line) is fixed, since here for illustration purpose we treat the full dataset as our population.

- For each sample, we take 40 data points randomly from the population, hence the **sample means** (dots) move around because each sample is different from each other (sampling variability). 

- The **confidence intervals** (horizontal bars) also change with each sample.

- As shown in the previous printout, about **95% of the intervals cover the population mean** — but a few miss it (shown in red).

❓<span style="color:purple;">**Quick test**</span>: Display a different subset of the confidence intervals, by modifying the code `subset = df_results.iloc[:40]` to `subset = df_results.iloc[60:100]` at the beginning, and check whether it matches your expectation.


❓<span style="color:purple;">**Quick test**</span>:
Suppose we use the same dataset, but now we compute **90%** confidence intervals instead of 95%. Do you expect **more intervals or fewer intervals** to miss the true mean?

<details> <summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;"> 
    With 90% confidence intervals, <b>more intervals will miss the true mean</b>. 
</p>

**Explanation**:

<p style="color:blue;"> 
A 90% confidence interval is narrower than a 95% confidence interval, so it is more likely to miss the true population mean. Over many repeated samples, we expect approximately 10% of the 90% confidence intervals to miss the true mean, compared with approximately 5% of the 95% confidence intervals. The observed percentages may vary because of random sampling variation.
</p>
    
</details>

## <span style="color:blue;"> Example 5: Bootstrap Confidence Interval </span>

The bootstrap approximates the sampling distribution by **resampling with replacement from the observed sample**. Bootstrap estimates how much the statistic (e.g. mean) from the observed sample would vary across samples of the same size. 

Bootstrap can estimate a confidence interval (CI) for many statistics — not just the mean: medians, proportions, regression coefficients, etc.

<p align="right">
  <img src="bootstrap_confidence_interval.png" alt="bootstrap_confidence_interval" width="400"/>
</p>

<span style="color:blue;"> 

<span style="color:blue;"> **Bootstrap the 95% confidence interval of the mean BMI** </span>

The code below estimates the confidence interval for the mean BMI by **bootstrapping**:

<div style="font-size: 16px; margin-left:15px;">
    
1) Begin with an initial sample of 100 BMI values. Here, we use the sample from Example 1.

2) Generate 1,000 bootstrap samples from the initial sample.

    - Each bootstrap sample contains 100 BMI values drawn with replacement.

    - Calculate the mean BMI for each bootstrap sample.

    - This produces a bootstrap distribution containing 1,000 sample means.

3) Use the 2.5th and 97.5th percentiles of the bootstrap distribution as the bounds of the 95% percentile bootstrap confidence interval.

</div>

In [ ]:
# Simulation settings
number_of_bootstrap_samples = 1000           # number of repeated sampling
confidence_level = 0.95

# Set a random seed for reproducibility
rng = np.random.default_rng(seed=2026)

# The result will hold the sample means
bootstrapped_sample_means = []

initial_sample = one_bmi_sample.copy()

# Generate bootstrap samples and calculate their means.
for _ in range(number_of_bootstrap_samples):
    # Take a random sample with replacement from the initial sample.
    bootstrap_sample = initial_sample.sample(n=len(initial_sample),      # sample size
                                             replace=True,               # sampling with replacement 
                                             random_state=rng            # random seed for reproducibility
                                           )    

    bmi_sample_mean = bootstrap_sample.mean()
    bootstrapped_sample_means.append(bmi_sample_mean)

bootstrap_means = pd.Series(bootstrapped_sample_means)

# Calculate the 2.5th and 97.5th percentiles for the 95% confidence interval
confidence_level = 0.95
alpha = 1 - confidence_level
# The 95% confidence interval corresponds to the middle 95% of the bootstrap 
# distribution, hence we drop 2.5% at the two tails. The below code translates to: 
# lower_bound, upper_bound = results.quantile([0.025, 0.975])
bootstrap_lower_bound, bootstrap_upper_bound = bootstrap_means.quantile([alpha / 2, 1 - alpha / 2])

print(f"Observed sample mean: {initial_sample.mean():.2f}")
print(f"Mean of bootstrap means: {bootstrap_means.mean():.2f}")
print(f"95% percentile bootstrap CI: " f"[{bootstrap_lower_bound:.2f}, {bootstrap_upper_bound:.2f}]")

<span style="color:blue;"> **Visualize the bootstrapped sample means and the 95% confidence interval** </span>

In [ ]:
# Convert the pandas Series of bootstrap means to a NumPy array.
# Each value represents the mean calculated from one bootstrap sample.
bootstrap_means = bootstrap_means.to_numpy()

# Create a histogram showing the bootstrap distribution of the sample mean.
# ax.hist() returns the bin frequencies, bin boundaries, and bar objects.
fig, ax = plt.subplots(figsize=(8, 4.5))
counts, bin_edges, bars = ax.hist(bootstrap_means, bins=25, color="#9DC3E6",
                                  edgecolor="#4A90E2", linewidth=0.8, zorder=2)

# Color the histogram bars in the two tails red.
# The center of each bin determines whether the bar belongs to the
# central 95% or one of the two 2.5% tails.
for left_edge, right_edge, bar in zip(bin_edges[:-1], bin_edges[1:], bars):
    bin_center = (left_edge + right_edge) / 2

    if bin_center < bootstrap_lower_bound or bin_center > bootstrap_upper_bound:
        bar.set_facecolor("#FF9999")
        bar.set_edgecolor("red")

# Use the tallest histogram bar as a reference for positioning the
# endpoint labels, tail percentages, and confidence-interval label.
maximum_height = counts.max()

# Add space above the histogram for the percentile-bound labels.
# The space below zero is used only for the green CI label; it does
# not represent negative frequencies.
upper_space = maximum_height * 1.30
lower_space = -maximum_height * 0.18
ax.set_ylim(lower_space, upper_space)

# Add a small amount of horizontal space on both sides of the histogram
# so the 2.5% tail labels are not too close to the plot boundaries.
data_range = bootstrap_means.max() - bootstrap_means.min()
x_padding = data_range * 0.06
ax.set_xlim(bootstrap_means.min() - x_padding, bootstrap_means.max() + x_padding)

# Save the horizontal limits because they are used below to position
# the two tail labels.
x_min, x_max = ax.get_xlim()

# Draw dashed red lines at the 2.5th and 97.5th percentiles.
# vlines() starts at y=0, so the lines do not extend into the space
# reserved below the histogram.
ax.vlines([bootstrap_lower_bound, bootstrap_upper_bound], ymin=0, ymax=maximum_height * 1.10,
          colors="red", linestyles="--", linewidth=2.5, zorder=4)

# Define one common style for the lower- and upper-bound labels.
# The white background prevents histogram grid lines from making
# the text difficult to read.
label_style = {"color": "red", "ha": "center", "va": "center",
               "fontsize": 9, "fontweight": "bold",
               "bbox": {
                   "facecolor": "white", "edgecolor": "none",
                   "alpha": 0.90, "pad": 2
                   }}

# Display each percentile value above its corresponding dashed line.
# The labels are placed above the line endpoints to avoid overlap.
ax.text(bootstrap_lower_bound, maximum_height * 1.18, f"lower bound\n{bootstrap_lower_bound:.2f}", **label_style)
ax.text(bootstrap_upper_bound, maximum_height * 1.18, f"upper bound\n{bootstrap_upper_bound:.2f}", **label_style)

# Place the 2.5% labels approximately in the middle of the two tails.
ax.text((x_min + bootstrap_lower_bound) / 2, maximum_height * 0.84, "2.5%",
        color="red", ha="center", fontsize=10, fontweight="bold"
        )
ax.text((bootstrap_upper_bound + x_max) / 2, maximum_height * 0.84, "2.5%",
        color="red", ha="center", fontsize=10, fontweight="bold")

# Draw a thick green line between the percentile endpoints.
# This line represents the middle 95% of the bootstrap distribution.
ax.hlines(y=0, xmin=bootstrap_lower_bound, xmax=bootstrap_upper_bound, color="#70AD47", linewidth=5, zorder=5)

# Label the green confidence-interval line below the histogram.
ax.text((bootstrap_lower_bound + bootstrap_upper_bound) / 2, -maximum_height * 0.08,
         "95% percentile bootstrap CI", color="#548235",
        ha="center", va="center", fontsize=10, fontweight="bold")

# Add descriptive labels for the axes and the figure.
# B is the number of bootstrap samples used to form the distribution.
ax.set_xlabel("Bootstrap sample mean BMI", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.set_title(f"Bootstrap Distribution of Mean BMI (B={len(bootstrap_means):,})",
             fontsize=11, pad=12)

# Add subtle horizontal grid lines behind the histogram to make the
# bar heights easier to compare.
ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.35, zorder=0)

# Remove the top and right borders to produce a cleaner figure.
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Adjust spacing and display the completed plot.
plt.tight_layout()
plt.show()

## <span style="color:blue;"> Example 6: Compare the formula-based CI and the bootstrap CI</span>

The code below compares the formual-based CI obtained in **Example 1** against the bootstrap CI obtained in **Example 5**.

In [ ]:
# Put the confidence interval obtained with both methods in the same table. 
# They estimate the same target at the same confidence level (95%), but 
# use different interval construction methods.
comparison = pd.DataFrame([
    {
        "Method": "Formula-based CI",
        "Lower": result["lower bound"].iloc[0],
        "Upper": result["upper bound"].iloc[0],
    },
    {
        "Method": "Bootstrap CI",
        "Lower": bootstrap_lower_bound,
        "Upper": bootstrap_upper_bound,
    },
])

comparison.round(3)

❓<span style="color:purple;">**Quick test**</span>: Based on the above output, what observation can you make?

<details>
<summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;">
The two methods produce very similar 95% confidence intervals that overlap almost completely.
</p>

**Explanation**:

<p style="color:blue;">
The formula-based interval is \([25.430,\ 27.160]\), while the bootstrap interval is \([25.474,\ 27.165]\). Therefore, both methods provide a similar estimate of the uncertainty around the sample mean and lead to the same practical conclusion about the population mean BMI.
</p>

<p style="color:blue;">
The intervals are not exactly identical because they are constructed differently: the formula-based method uses the \(t\)-distribution, whereas the percentile bootstrap method uses percentiles of the bootstrap distribution.
</p>

</details>

# 🏠 Assignment Questions <span style="color:blue;"> [20 points] </span>
For the assignment, we will mainly focus on **blood pressure**, it corresponds to the **bp** column in the dataset.

In [ ]:
blood_pressure = df["bp"]

plt.figure(figsize=(4, 3))

# Plot the a histgram of the blood pressure
# bins controls the histogram resolution; kde=True adds a smooth density
# estimate to help reveal distribution shape.
sns.histplot(blood_pressure, bins=10, kde=True, color="#5B9BD5")
plt.title("Distribution of Blood Pressure")
plt.xlabel("Blood pressure")
plt.ylabel("Count")

# Adjust spacing so titles and labels do not overlap.
plt.tight_layout()
plt.show()

## <span style="color:blue;"> [4 points] </span>Question-1: Compute the 99% CI for mean **blood pressure** and the **margin of error** using the formula. Print the mean BP, the estimated standard error, the 99% CI as well as the margin of error.

#### **Q1-1: Draw one random sample which contains 50 observations, compute and print out the sample mean** 

In [ ]:
# insert your code here

####  **Q1-2: Compute and print the estimated standard error (SE)** 

In [ ]:
# insert your code here

#### **Q1-3: Compute and print the margin of error and the 99% confidence interval of the sample mean** 

In [ ]:
# insert your code here

#### **Q1-4: Build a dataframe to display the different values** 

In [ ]:
# insert your code here

## <span style="color:blue;"> [3 points] </span> Question-2: Compute and vsualize 90%, 95%, 99% CIs for the mean BP in a single figure.

#### **Q2-1: which confidence interval would be the widest? Explain your reasoning and justify your answer without doing the exact calculation.**

----
<div style="color:blue">
    
**Answer**: insert your answer here


</div>

----


#### **Q2-2: Compute the 90%, 95%, 99% CIs for the mean BP as well as the Margin of Error for each confidence level. Diplay the results in a well-formatted Pandas dataframe.**

In [ ]:
#insert your code here

#### **Q2-3: Visualize the 90%, 95%, 99% CIs for the mean BP. Include a verticle line corresponding to the mean BP as well.**

In [ ]:
#insert your code here

## <span style="color:blue;"> [3 points] </span> Question-3: Bootstrap the 99% Confidence Interval for mean blood pressure

**Requirement**: use the same random sample as in Q1-1, and use 4000 for the number of repeated samplings to obtain better stability.

#### **Q3-1: Bootstrap the 99% confidence interval for the mean blood pressure and print it**

In [ ]:
# insert your code here

## <span style="color:blue;"> [4 points] </span>  Question-4: compare the formula-based CI and the bootstrap CI for mean blood pressure.

#### **Q4-1: build a dataframe which includes the lower bound and upper bound for the formula-based CI and bootstrap CI, and display it.**

In [ ]:
# insert your code here

#### **Q4-2: if we want to a claim that "the population mean blood pressure is below 100, with a 99% confidence level". Does the formula-based CI and bootstrap CI support the claim or not. Explain your answer.**

----
<div style="color:blue">
    
**Answer**: insert your answer here


</div>

----


## <span style="color:blue;"> [6 points] </span> Question 5: take a random sample which contains 30 observations for blood pressure. Use it as a baseline and compute the 90% confidence interval. Take two other random samples, whose width of confidence interval should be approximately one-half and one-third of the width of the baseline sample.

#### **Q5-1: Determine the sample size for the other two random samples. Justify your answer.**

----
<div style="color:blue">
    
**Answer**: insert your answer here


</div>

----


#### **Q5-2: compute the CI for 3 different sample sizes for 90% confidence level**

In [ ]:
# insert your code here

#### **Q5-3: Visualize the 90% CIs for the mean blood pressure for each sample size in a single figure.** 

In [ ]:
# insert your code here